# 04 — Label Quality Mining

Goal: find label/data issues that cap mAP before changing hyperparameters.

What to look for:

- **Tiny boxes**: hard to localize; check if labels are consistent and objects are truly classifiable.
- **Huge or extreme aspect boxes**: often loose boxes, truncated vehicles, or annotation mistakes.
- **Edge boxes**: vehicles cut by image border can create inconsistent extents.
- **Crowded images**: missing one box can create many false positives during review.
- **Class imbalance**: rare classes need targeted cleanup because AP is averaged by class.

This notebook does not change labels. Use it to choose 3LC review filters and revision targets.


In [ ]:
from pathlib import Path
import os
import random
import sys


def find_repo_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "config" / "competition.yaml").is_file():
            return path
    raise FileNotFoundError("Could not find repo root containing config/competition.yaml")


REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))
print(REPO_ROOT)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from matplotlib.patches import Rectangle

from ua_detrac_starter.config import config_path, dataset_split_path, load_config, load_yaml

cfg, _ = load_config("config/competition.yaml")
dataset_yaml = config_path(cfg, "dataset_yaml", "config/dataset.yaml")
dataset_cfg = load_yaml(dataset_yaml)

train_images = dataset_split_path(dataset_yaml, dataset_cfg, "train")
val_images = dataset_split_path(dataset_yaml, dataset_cfg, "val")
CLASS_NAMES = {int(k): v for k, v in dataset_cfg.get("names", {}).items()}
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

print("train:", train_images)
print("val:", val_images)


## Build A Label Table

Comments while reading:

- Do not only look at aggregate counts. mAP is sensitive to a small number of bad labels in rare classes.
- Keep examples tied to `image_path` and `label_path` so they can be found quickly in 3LC or on disk.


In [ ]:
def list_images(directory: Path) -> list[Path]:
    if not directory.is_dir():
        return []
    return sorted(path for path in directory.iterdir() if path.suffix.lower() in IMAGE_EXTS)


def parse_split(images_dir: Path, split: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    rows = []
    image_rows = []
    labels_dir = images_dir.parent / "labels"
    for image_path in list_images(images_dir):
        image = Image.open(image_path)
        image_width, image_height = image.size
        label_path = labels_dir / f"{image_path.stem}.txt"
        raw_lines = label_path.read_text(encoding="utf-8").splitlines() if label_path.is_file() else []
        lines = [line.strip() for line in raw_lines if line.strip()]
        image_rows.append({
            "split": split,
            "image_id": image_path.stem,
            "image_path": str(image_path),
            "label_path": str(label_path),
            "image_width": image_width,
            "image_height": image_height,
            "box_count": len(lines),
            "has_label_file": label_path.is_file(),
        })
        for line_number, line in enumerate(lines, start=1):
            parts = line.split()
            if len(parts) != 5:
                continue
            class_id = int(parts[0])
            x_center, y_center, width, height = [float(value) for value in parts[1:]]
            aspect = width / height if height > 0 else np.inf
            rows.append({
                "split": split,
                "image_id": image_path.stem,
                "image_path": str(image_path),
                "label_path": str(label_path),
                "line_number": line_number,
                "class_id": class_id,
                "class_name": CLASS_NAMES.get(class_id, str(class_id)),
                "x_center": x_center,
                "y_center": y_center,
                "width": width,
                "height": height,
                "area": width * height,
                "aspect": aspect,
                "x1": x_center - width / 2,
                "y1": y_center - height / 2,
                "x2": x_center + width / 2,
                "y2": y_center + height / 2,
                "touches_edge": x_center - width / 2 <= 0.01 or y_center - height / 2 <= 0.01 or x_center + width / 2 >= 0.99 or y_center + height / 2 >= 0.99,
                "image_width": image_width,
                "image_height": image_height,
            })
    return pd.DataFrame(rows), pd.DataFrame(image_rows)


train_boxes, train_image_rows = parse_split(train_images, "train")
val_boxes, val_image_rows = parse_split(val_images, "val")
boxes = pd.concat([train_boxes, val_boxes], ignore_index=True)
images = pd.concat([train_image_rows, val_image_rows], ignore_index=True)

print(f"images: {len(images):,}")
print(f"boxes: {len(boxes):,}")
display(pd.crosstab(boxes["class_name"], boxes["split"], margins=True) if not boxes.empty else boxes)
images.head()


## Quantify Suspicious Labels

Use these buckets to prioritize manual review. They are not automatically wrong.

- `tiny`: likely to hurt recall and localization.
- `huge`: often loose labels or non-standard scenes.
- `extreme_aspect`: trailers, buses, or annotation mistakes; inspect examples.
- `edge_touch`: border truncation; label policy must be consistent.
- `crowded`: high-value images because one cleanup pass can fix many boxes.


In [ ]:
if boxes.empty:
    print("No boxes loaded.")
else:
    tiny_area = boxes["area"].quantile(0.05)
    huge_area = boxes["area"].quantile(0.99)
    boxes["tiny"] = boxes["area"] <= tiny_area
    boxes["huge"] = boxes["area"] >= huge_area
    boxes["extreme_aspect"] = (boxes["aspect"] >= 4.0) | (boxes["aspect"] <= 0.25)

    images["crowded"] = images["box_count"] >= images["box_count"].quantile(0.95)

    issue_summary = pd.DataFrame([
        {"bucket": "tiny", "count": int(boxes["tiny"].sum()), "lookout": "small or distant vehicles; check consistency and classifiability"},
        {"bucket": "huge", "count": int(boxes["huge"].sum()), "lookout": "loose boxes, close vehicles, or bad normalization"},
        {"bucket": "extreme_aspect", "count": int(boxes["extreme_aspect"].sum()), "lookout": "very flat/tall boxes; check if extent is correct"},
        {"bucket": "edge_touch", "count": int(boxes["touches_edge"].sum()), "lookout": "truncated vehicles; ensure border policy is consistent"},
        {"bucket": "crowded_images", "count": int(images["crowded"].sum()), "lookout": "missing boxes in dense scenes"},
        {"bucket": "missing_label_files", "count": int((~images["has_label_file"]).sum()), "lookout": "images with no label file"},
        {"bucket": "empty_label_files", "count": int((images["box_count"] == 0).sum()), "lookout": "empty scenes or missing annotations"},
    ])
    display(issue_summary)


In [ ]:
if not boxes.empty:
    fig, axes = plt.subplots(2, 2, figsize=(13, 8))
    pd.crosstab(boxes["class_name"], boxes["split"]).plot(kind="bar", ax=axes[0, 0], title="Class balance")
    boxes["area"].clip(upper=boxes["area"].quantile(0.99)).plot(kind="hist", bins=50, ax=axes[0, 1], title="Box area distribution")
    boxes["aspect"].clip(upper=8).plot(kind="hist", bins=50, ax=axes[1, 0], title="Aspect ratio distribution")
    images["box_count"].plot(kind="hist", bins=40, ax=axes[1, 1], title="Boxes per image")
    for ax in axes.flat:
        ax.grid(True, alpha=0.2)
    plt.tight_layout()


## Inspect Suspicious Examples

Change `REVIEW_BUCKET` and rerun. The goal is to decide whether to create a 3LC revision, not to prove the model is wrong.

Review comments:

- If many examples in a bucket are valid, do not waste time fixing them.
- If one class dominates a suspicious bucket, create a targeted 3LC filter for that class.
- If labels are inconsistent, fix the policy first, then edit labels.


In [ ]:
def draw_label_review(rows: pd.DataFrame, max_images: int = 6):
    if rows.empty:
        print("No rows to review.")
        return
    image_ids = rows["image_id"].drop_duplicates().head(max_images).tolist()
    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    for ax, image_id in zip(axes.flat, image_ids):
        image_rows = boxes[boxes["image_id"] == image_id]
        image_path = Path(image_rows["image_path"].iloc[0])
        image = Image.open(image_path).convert("RGB")
        image_width, image_height = image.size
        ax.imshow(image)
        ax.set_title(f"{image_rows['split'].iloc[0]} / {image_path.name}")
        ax.axis("off")
        for _, row in image_rows.iterrows():
            x = row.x1 * image_width
            y = row.y1 * image_height
            w = row.width * image_width
            h = row.height * image_height
            color = "red" if row.name in set(rows.index) else "lime"
            ax.add_patch(Rectangle((x, y), w, h, fill=False, edgecolor=color, linewidth=2))
            ax.text(x, max(0, y - 4), row.class_name, color="black", backgroundcolor=color, fontsize=8)
    for ax in axes.flat[len(image_ids):]:
        ax.axis("off")
    plt.tight_layout()


REVIEW_BUCKET = "tiny"  # tiny, huge, extreme_aspect, edge_touch, crowded

if boxes.empty:
    print("No boxes loaded.")
elif REVIEW_BUCKET == "tiny":
    review_rows = boxes[boxes["tiny"]].sort_values("area")
elif REVIEW_BUCKET == "huge":
    review_rows = boxes[boxes["huge"]].sort_values("area", ascending=False)
elif REVIEW_BUCKET == "extreme_aspect":
    review_rows = boxes[boxes["extreme_aspect"]].sort_values("aspect", ascending=False)
elif REVIEW_BUCKET == "edge_touch":
    review_rows = boxes[boxes["touches_edge"]]
elif REVIEW_BUCKET == "crowded":
    crowded_ids = set(images[images["crowded"]]["image_id"])
    review_rows = boxes[boxes["image_id"].isin(crowded_ids)].sort_values("image_id")
else:
    raise ValueError("Unknown REVIEW_BUCKET")

print(f"review bucket={REVIEW_BUCKET}, boxes={len(review_rows):,}")
display(review_rows.head(20))
draw_label_review(review_rows)


## Turn Label Mining Into A 3LC Action

Recommended action format:

```text
Revision: fix_tiny_val_car_boxes_v1
Filter: split=val, class=car, area <= q05
Decision: tighten boxes where vehicle extent is clear; ignore ambiguous tiny objects
Expected mAP effect: fewer localization errors and false negatives for car
```

Only create a revision when the examples show a consistent problem. Random one-off edits are hard to justify and may not move mAP.
